In [1]:
from __future__ import annotations

import importlib.util
import json
import os
import random
import sys
from pathlib import Path

import numpy as np
import polars as pl
import torch


# Remote kernel이면 여기에 서버 기준 repo 절대경로를 넣을 수 있습니다.
# Example: REPO_ROOT_OVERRIDE = Path("/home/ubuntu/ts_forecaster_lib")
# repo clone이 서버에 없고 modeling_module만 설치되어 있어도 notebook는 동작하도록 구성합니다.
REPO_ROOT_OVERRIDE = None


def _looks_like_repo(path: Path) -> bool:
    return (path / "pyproject.toml").exists() and (path / "src").exists()


def _iter_named_repo_candidates(root: Path, repo_name: str = "ts_forecaster_lib", max_depth: int = 4):
    if not root.exists() or not root.is_dir():
        return

    try:
        root_resolved = root.resolve()
    except Exception:
        root_resolved = root

    stack = [(root_resolved, 0)]
    while stack:
        current, depth = stack.pop()
        if current.name == repo_name:
            yield current
        if depth >= max_depth:
            continue
        try:
            children = list(current.iterdir())
        except Exception:
            continue
        for child in children:
            if child.is_dir() and not child.name.startswith('.'):
                stack.append((child, depth + 1))


def find_repo_root(start: Path, explicit_repo_root: Path | None = None) -> Path | None:
    env_repo_root = os.environ.get("TS_FORECASTER_REPO_ROOT")

    candidates = []
    if explicit_repo_root is not None:
        candidates.append(Path(explicit_repo_root).expanduser().resolve())
    if env_repo_root:
        candidates.append(Path(env_repo_root).expanduser().resolve())
    candidates.extend([start, *start.parents])

    home = Path.home()
    common_roots = [
        home,
        home / "workspace",
        home / "workspaces",
        home / "projects",
        home / "PycharmProjects",
        Path("/workspace"),
        Path("/workspaces"),
        Path("/home"),
        Path("/root"),
    ]
    for root in common_roots:
        candidates.extend(_iter_named_repo_candidates(root))

    seen = set()
    for candidate in candidates:
        if candidate in seen:
            continue
        seen.add(candidate)
        if _looks_like_repo(candidate):
            return candidate

    return None


def resolve_import_paths() -> tuple[Path | None, Path | None]:
    repo_root = find_repo_root(Path.cwd().resolve(), explicit_repo_root=REPO_ROOT_OVERRIDE)
    src_root = repo_root / "src" if repo_root is not None else None

    if src_root is not None and str(src_root) not in sys.path:
        sys.path.insert(0, str(src_root))

    spec = importlib.util.find_spec("modeling_module")
    if spec is None or spec.origin is None:
        raise RuntimeError(
            "Could not import modeling_module. Either set REPO_ROOT_OVERRIDE to the server repo path, "
            "set TS_FORECASTER_REPO_ROOT, or install the package on the remote environment."
        )

    module_init = Path(spec.origin).resolve()
    module_root = module_init.parent

    if repo_root is None and module_root.parent.name == "src":
        repo_root = module_root.parent.parent
        src_root = repo_root / "src"

    return repo_root, src_root


NOTEBOOK_DIR = Path.cwd().resolve()
REPO_ROOT, SRC_ROOT = resolve_import_paths()

from modeling_module import (
    ArtifactConfig,
    DataColumnConfig,
    DataRequest,
    DataWindowConfig,
    ExogenousConfig,
    LoaderConfig,
    RuntimeConfig,
    SSLConfig,
    TrainRequest,
    TrainerConfig,
    build_dataloader,
    build_dataset,
    load_predictor,
    train,
)
from modeling_module.utils.device import select_default_device


SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEFAULT_DEVICE, DEFAULT_DEVICE_DIAGNOSTIC = select_default_device()

print("REPO_ROOT:", REPO_ROOT)
print("SRC_ROOT :", SRC_ROOT)
print("PYTHON   :", sys.executable)
print("TORCH    :", torch.__version__)
print("DEVICE   :", DEFAULT_DEVICE)
if DEFAULT_DEVICE_DIAGNOSTIC:
    print("DEVICE_NOTE:", DEFAULT_DEVICE_DIAGNOSTIC)


REPO_ROOT: /home/leekwanhyeong/workspace/ts_forecaster_lib
SRC_ROOT : /home/leekwanhyeong/workspace/ts_forecaster_lib/src
PYTHON   : /home/leekwanhyeong/miniconda3/envs/ai_env/bin/python
TORCH    : 2.12.0.dev20260408+cu128
DEVICE   : cuda


In [2]:
# Example:
# if REPO_ROOT is not None:
#     TARGET_SOURCE = REPO_ROOT / "raw_data" / "exports" / "tb_master_target.parquet"
#     EXO_SOURCE = REPO_ROOT / "raw_data" / "exports" / "tb_master_exo.parquet"

DATA_ROOT = REPO_ROOT / "raw_data" / "master" if REPO_ROOT is not None else None
TARGET_SOURCE = DATA_ROOT / "tb_master_target.parquet" if DATA_ROOT is not None else None
EXO_SOURCE = DATA_ROOT / "tb_master_exo.parquet" if DATA_ROOT is not None else None

FREQ = "weekly"
ID_COL = "oper_part_no"
DATE_COL = "demand_dt"
Y_COL = "demand_qty"

# tb_master_exo에 y가 없으면 tb_master_target의 y를 join합니다.
JOIN_TARGET_INTO_EXO = True

PAST_EXO_CONT_COLS = [
    "sin_annual",
    "cos_annual",
    "sin_semi",
    "cos_semi",
    "sin_quarter",
    "cos_quarter",
    "weather_index",
    "macro_index",
    "promo_strength",
    "part_len",
    "week_of_year",
]
FUTURE_EXO_CONT_COLS = [
    "sin_annual",
    "cos_annual",
    "sin_semi",
    "cos_semi",
    "sin_quarter",
    "cos_quarter",
    "weather_index",
    "macro_index",
    "promo_strength",
    "week_of_year",
    "promo_flag",
    "supply_outage_flag",
    "peak_season_flag",
    "is_year_start",
    "is_year_end",
    "is_q_start",
    "is_q_end",
]
PAST_EXO_CAT_COLS = []
# Optional categorical candidate: ["part_group_id"]

LOOKBACK = 52
HORIZON = 27
BATCH_SIZE = 16
MAX_IDS = 32
MIN_OBSERVED_TARGET_ROWS = LOOKBACK + HORIZON

TRAIN_EPOCHS = 1
TRAIN_LR = 1e-3
TRAIN_DEVICE = DEFAULT_DEVICE

ENDO_MODELS = ["patchtst_base"]
EXO_MODELS = ["patchtst_base"]
# ExoTST를 쓰고 싶으면 보통 둘 다 필요합니다.
# EXO_MODELS = ["exotst_base"]

ARTIFACT_BASE = REPO_ROOT if REPO_ROOT is not None else NOTEBOOK_DIR
ARTIFACT_ROOT = ARTIFACT_BASE / "artifacts" / "notebook_manual_checks"
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

config_snapshot = {
    "TARGET_SOURCE": str(TARGET_SOURCE) if TARGET_SOURCE is not None else None,
    "EXO_SOURCE": str(EXO_SOURCE) if EXO_SOURCE is not None else None,
    "FREQ": FREQ,
    "ID_COL": ID_COL,
    "DATE_COL": DATE_COL,
    "Y_COL": Y_COL,
    "PAST_EXO_CONT_COLS": PAST_EXO_CONT_COLS,
    "FUTURE_EXO_CONT_COLS": FUTURE_EXO_CONT_COLS,
    "PAST_EXO_CAT_COLS": PAST_EXO_CAT_COLS,
    "LOOKBACK": LOOKBACK,
    "HORIZON": HORIZON,
    "BATCH_SIZE": BATCH_SIZE,
    "ENDO_MODELS": ENDO_MODELS,
    "EXO_MODELS": EXO_MODELS,
    "TRAIN_DEVICE": TRAIN_DEVICE,
}
print(json.dumps(config_snapshot, indent=2, ensure_ascii=False))


{
  "TARGET_SOURCE": "/home/leekwanhyeong/workspace/ts_forecaster_lib/raw_data/master/tb_master_target.parquet",
  "EXO_SOURCE": "/home/leekwanhyeong/workspace/ts_forecaster_lib/raw_data/master/tb_master_exo.parquet",
  "FREQ": "weekly",
  "ID_COL": "oper_part_no",
  "DATE_COL": "demand_dt",
  "Y_COL": "demand_qty",
  "PAST_EXO_CONT_COLS": [
    "sin_annual",
    "cos_annual",
    "sin_semi",
    "cos_semi",
    "sin_quarter",
    "cos_quarter",
    "weather_index",
    "macro_index",
    "promo_strength",
    "part_len",
    "week_of_year"
  ],
  "FUTURE_EXO_CONT_COLS": [
    "sin_annual",
    "cos_annual",
    "sin_semi",
    "cos_semi",
    "sin_quarter",
    "cos_quarter",
    "weather_index",
    "macro_index",
    "promo_strength",
    "week_of_year",
    "promo_flag",
    "supply_outage_flag",
    "peak_season_flag",
    "is_year_start",
    "is_year_end",
    "is_q_start",
    "is_q_end"
  ],
  "PAST_EXO_CAT_COLS": [],
  "LOOKBACK": 52,
  "HORIZON": 27,
  "BATCH_SIZE": 16,
  "E

In [ ]:
tb_mst_oper_part = pl.read_parquet(f"{REPO_ROOT}/raw_data/raw/tb_mst_oper_part.parquet")
tb_dyn_sales_parts = pl.read_parquet(f"{REPO_ROOT}/raw_data/raw/tb_dyn_sales_parts.parquet")
tb_dyn_demand_dtl = pl.read_parquet(f"{REPO_ROOT}/raw_data/raw/tb_dyn_demand_dtl.parquet")

tb_dyn_sales_parts

# 산출식
# 유상 보증 대수, 무상 보증 대수
# 경과주차별 불량률


In [ ]:
try:
    from xgboost import XGBRegressor
except ImportError as exc:
    raise ImportError(
        "xgboost is not installed in this kernel. Run `%pip install xgboost` and rerun this section."
    ) from exc

import matplotlib.pyplot as plt
from datetime import date

XGB_TARGET_COL = "demand_qty"
XGB_MAX_PARTS = 5_000
XGB_SAMPLE_ROWS = 500_000
XGB_TEST_WEEKS = 26
XGB_RANDOM_STATE = SEED
XGB_N_ESTIMATORS = 300
XGB_MAX_DEPTH = 6
XGB_LEARNING_RATE = 0.05
XGB_SUBSAMPLE = 0.8
XGB_COLSAMPLE_BYTREE = 0.8
XGB_TOP_K = 20


def yearweek_to_monday(yearweek: int) -> date:
    year = int(yearweek) // 100
    week = int(yearweek) % 100
    return date.fromisocalendar(year, week, 1)


def build_dense_week_index(*arrays) -> dict[int, int]:
    all_weeks = sorted({int(v) for arr in arrays for v in arr})
    start = min(yearweek_to_monday(v) for v in all_weeks)
    end = max(yearweek_to_monday(v) for v in all_weeks)

    dense_weeks: list[int] = []
    cur = start
    while cur <= end:
        iso = cur.isocalendar()
        dense_weeks.append((iso.year * 100) + iso.week)
        cur = date.fromordinal(cur.toordinal() + 7)
    return {w: i for i, w in enumerate(dense_weeks)}


print({
    "XGB_TARGET_COL": XGB_TARGET_COL,
    "XGB_MAX_PARTS": XGB_MAX_PARTS,
    "XGB_SAMPLE_ROWS": XGB_SAMPLE_ROWS,
    "XGB_TEST_WEEKS": XGB_TEST_WEEKS,
})


In [ ]:
selected_parts = tb_dyn_demand_dtl.select("oper_part_no").unique().sort("oper_part_no")
if XGB_MAX_PARTS is not None:
    selected_parts = selected_parts.head(XGB_MAX_PARTS)

selected_part_nos = selected_parts["oper_part_no"].to_list()

oper_base = (
    tb_mst_oper_part
    .filter(pl.col("oper_part_no").is_in(selected_part_nos))
    .select(["oper_part_no", "part_fam_cd", "demand_start_dt", "warranty"])
    .with_columns((pl.col("warranty") * 52 / 12).round(0).cast(pl.Int64).alias("warranty_weeks"))
)

sales_base = (
    tb_dyn_sales_parts
    .filter(pl.col("oper_part_no").is_in(selected_part_nos))
    .group_by(["oper_part_no", "sales_dt"])
    .agg(pl.col("sales_qty").sum().alias("sales_qty_current_week"))
)

demand_base = (
    tb_dyn_demand_dtl
    .filter(pl.col("oper_part_no").is_in(selected_part_nos))
    .group_by(["oper_part_no", "demand_dt"])
    .agg(pl.col("demand_qty").sum().alias("demand_qty"))
)

week_to_idx = build_dense_week_index(
    oper_base["demand_start_dt"].to_list(),
    sales_base["sales_dt"].to_list(),
    demand_base["demand_dt"].to_list(),
)

oper_base = oper_base.with_columns(
    pl.col("demand_start_dt").replace_strict(week_to_idx).alias("start_wk_idx")
)

sales_weekly = (
    sales_base
    .join(oper_base.select(["oper_part_no"]), on="oper_part_no", how="inner")
    .with_columns(pl.col("sales_dt").replace_strict(week_to_idx).alias("wk_idx"))
    .drop("sales_dt")
    .sort(["oper_part_no", "wk_idx"])
)

sales_cum = (
    sales_weekly
    .with_columns(pl.col("sales_qty_current_week").cum_sum().over("oper_part_no").alias("cum_sales_total"))
    .select(["oper_part_no", "wk_idx", "cum_sales_total"])
    .sort(["oper_part_no", "wk_idx"])
)

demand_weekly = (
    demand_base
    .join(oper_base, on="oper_part_no", how="inner")
    .with_columns(pl.col("demand_dt").replace_strict(week_to_idx).alias("wk_idx"))
    .drop("demand_dt")
    .sort(["oper_part_no", "wk_idx"])
)

xgb_feature_table = (
    demand_weekly
    .join(
        sales_weekly.select(["oper_part_no", "wk_idx", "sales_qty_current_week"]),
        on=["oper_part_no", "wk_idx"],
        how="left",
    )
    .with_columns(pl.col("sales_qty_current_week").fill_null(0.0))
    .join_asof(
        sales_cum,
        on="wk_idx",
        by="oper_part_no",
        strategy="backward",
    )
    .with_columns(pl.col("cum_sales_total").fill_null(0.0))
)

sales_cum_prev = sales_cum.rename({"wk_idx": "cutoff_wk_idx", "cum_sales_total": "cum_sales_prev_warranty"}).sort(["oper_part_no", "cutoff_wk_idx"])

xgb_feature_table = (
    xgb_feature_table
    .with_columns((pl.col("wk_idx") - pl.col("warranty_weeks")).alias("cutoff_wk_idx"))
    .join_asof(
        sales_cum_prev,
        on="cutoff_wk_idx",
        by="oper_part_no",
        strategy="backward",
    )
    .with_columns(
        (pl.col("cum_sales_total") - pl.col("cum_sales_prev_warranty").fill_null(0.0))
        .clip(lower_bound=0.0)
        .alias("active_warranty_base")
    )
    .drop(["cutoff_wk_idx", "cum_sales_prev_warranty"])
)

for window, out_col in [(4, "recent_sales_4w"), (13, "recent_sales_13w"), (26, "recent_sales_26w")]:
    prev = sales_cum.rename({"wk_idx": "cutoff_wk_idx", "cum_sales_total": f"cum_sales_prev_{window}"}).sort(["oper_part_no", "cutoff_wk_idx"])
    xgb_feature_table = (
        xgb_feature_table
        .with_columns((pl.col("wk_idx") - window).alias("cutoff_wk_idx"))
        .join_asof(
            prev,
            on="cutoff_wk_idx",
            by="oper_part_no",
            strategy="backward",
        )
        .with_columns(
            (pl.col("cum_sales_total") - pl.col(f"cum_sales_prev_{window}").fill_null(0.0))
            .clip(lower_bound=0.0)
            .alias(out_col)
        )
        .drop(["cutoff_wk_idx", f"cum_sales_prev_{window}"])
    )

xgb_feature_table = (
    xgb_feature_table
    .with_columns([
        (pl.col("wk_idx") - pl.col("start_wk_idx")).alias("elapsed_week"),
        (pl.col("cum_sales_total") - pl.col("active_warranty_base")).clip(lower_bound=0.0).alias("post_warranty_base"),
    ])
    .filter(pl.col("elapsed_week") >= 0)
    .with_columns([
        pl.when(pl.col("warranty_weeks") > 0)
        .then(pl.col("elapsed_week") / pl.col("warranty_weeks"))
        .otherwise(0.0)
        .alias("warranty_progress"),
        (pl.col("warranty_weeks") - pl.col("elapsed_week")).alias("weeks_to_warranty_end"),
        pl.when(pl.col("active_warranty_base") > 0)
        .then(pl.col("demand_qty") / pl.col("active_warranty_base"))
        .otherwise(0.0)
        .alias("repair_incidence_rate"),
    ])
)

feature_cols = [
    "part_fam_cd",
    "warranty",
    "warranty_weeks",
    "elapsed_week",
    "warranty_progress",
    "weeks_to_warranty_end",
    "sales_qty_current_week",
    "cum_sales_total",
    "active_warranty_base",
    "post_warranty_base",
    "recent_sales_4w",
    "recent_sales_13w",
    "recent_sales_26w",
]

if XGB_TARGET_COL not in xgb_feature_table.columns:
    raise KeyError(f"XGB_TARGET_COL={XGB_TARGET_COL!r} not found. Available target candidates: demand_qty, repair_incidence_rate")

xgb_feature_table = xgb_feature_table.select(["oper_part_no", "wk_idx", XGB_TARGET_COL, *feature_cols])

print("feature table shape:", xgb_feature_table.shape)
print("n_parts:", xgb_feature_table.select(pl.col("oper_part_no").n_unique()).item())
print("target summary:")
print(xgb_feature_table.select([
    pl.col(XGB_TARGET_COL).mean().alias("mean"),
    pl.col(XGB_TARGET_COL).median().alias("median"),
    pl.col(XGB_TARGET_COL).max().alias("max"),
]))

xgb_feature_table.head()


In [ ]:
if XGB_SAMPLE_ROWS is not None and xgb_feature_table.height > XGB_SAMPLE_ROWS:
    model_df = xgb_feature_table.sample(n=XGB_SAMPLE_ROWS, seed=XGB_RANDOM_STATE, shuffle=True)
else:
    model_df = xgb_feature_table.clone()

split_wk_idx = int(model_df["wk_idx"].max()) - XGB_TEST_WEEKS
train_df = model_df.filter(pl.col("wk_idx") <= split_wk_idx)
test_df = model_df.filter(pl.col("wk_idx") > split_wk_idx)

if train_df.is_empty() or test_df.is_empty():
    raise RuntimeError("Train/Test split is empty. Reduce XGB_TEST_WEEKS or increase the available data range.")

train_X = train_df.select(feature_cols).to_dummies(columns=["part_fam_cd"])
test_X = test_df.select(feature_cols).to_dummies(columns=["part_fam_cd"])
all_feature_names = sorted(set(train_X.columns) | set(test_X.columns))


def align_feature_frame(df: pl.DataFrame, columns: list[str]) -> pl.DataFrame:
    missing_exprs = [pl.lit(0.0).alias(col) for col in columns if col not in df.columns]
    if missing_exprs:
        df = df.with_columns(missing_exprs)
    return df.select(columns).with_columns([pl.col(col).cast(pl.Float32) for col in columns])


train_X = align_feature_frame(train_X, all_feature_names)
test_X = align_feature_frame(test_X, all_feature_names)

y_train = train_df[XGB_TARGET_COL].to_numpy()
y_test = test_df[XGB_TARGET_COL].to_numpy()

xgb_model = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=XGB_N_ESTIMATORS,
    max_depth=XGB_MAX_DEPTH,
    learning_rate=XGB_LEARNING_RATE,
    subsample=XGB_SUBSAMPLE,
    colsample_bytree=XGB_COLSAMPLE_BYTREE,
    random_state=XGB_RANDOM_STATE,
    tree_method="hist",
    n_jobs=os.cpu_count() or 4,
)

xgb_model.fit(
    train_X.to_numpy(),
    y_train,
    eval_set=[(test_X.to_numpy(), y_test)],
    verbose=False,
)

y_pred = xgb_model.predict(test_X.to_numpy())
mae = float(np.mean(np.abs(y_test - y_pred)))
rmse = float(np.sqrt(np.mean((y_test - y_pred) ** 2)))

split_map = xgb_model.get_booster().get_score(importance_type="weight")
gain_map = xgb_model.get_booster().get_score(importance_type="gain")
importance_df = (
    pl.DataFrame({
        "feature": all_feature_names,
        "feature_importance_": xgb_model.feature_importances_,
        "split_importance": [float(split_map.get(f"f{i}", 0.0)) for i in range(len(all_feature_names))],
        "gain_importance": [float(gain_map.get(f"f{i}", 0.0)) for i in range(len(all_feature_names))],
    })
    .sort("feature_importance_", descending=True)
)

print({
    "train_rows": train_df.height,
    "test_rows": test_df.height,
    "split_wk_idx": split_wk_idx,
    "test_mae": round(mae, 6),
    "test_rmse": round(rmse, 6),
})

plt.figure(figsize=(10, max(4, int(XGB_TOP_K * 0.45))))
top_k = importance_df.head(XGB_TOP_K)
plt.barh(top_k["feature"].to_list()[::-1], top_k["feature_importance_"].to_list()[::-1])
plt.xlabel("feature_importance_")
plt.title(f"XGBoost Feature Importance ({XGB_TARGET_COL})")
plt.tight_layout()
plt.show()

importance_df.head(XGB_TOP_K)
